# Detektor v2: apakah foto CFPD menolong sepatu?

Tiga angka pada **satu set uji yang sama**, semuanya diukur di notebook ini:

1. **Baseline** — detektor yang sekarang terpasang (`garment_detector.tflite` di repo).
2. **Arm A** — `yolov8n` dilatih ulang pada Fashionpedia saja (subset 6.000 foto).
3. **Arm B** — model yang sama, data yang sama, **ditambah 1.880 foto CFPD**.

Kalau recall sepatu Arm B lebih tinggi daripada Arm A, data CFPD menolong. Kalau tidak, yang
lemah adalah pelokalan modelnya, bukan jumlah datanya — dan tidak perlu berburu dataset lagi.

Arm A ada supaya perbandingannya adil: dua-duanya mulai dari bobot ImageNet, jumlah epoch sama,
resolusi sama. Membandingkan langsung dengan model 40 epoch di GPU hanya akan mengukur bedanya
anggaran latihan, bukan bedanya data.

### Setelan sesi

1. Panel kanan → **Accelerator → GPU T4**.
2. Panel kanan → **Internet → On** (unduh Fashionpedia 3,48 GB).
3. **Add Input** → cari `colorful-fashion-dataset-for-object-detection` (dataset publik Kaggle,
   110 MB) → tambahkan. Tidak perlu mengunggah apa pun.

### Yang dihasilkan

`report.md` berisi tabel per Category untuk ketiga model, plus `arms.json` dan dua confusion
matrix. Semua dibungkus `gayaku_detect_v2.zip` di `/kaggle/working`.

### Catatan lisensi

CFPD adalah dataset riset (Kaggle menandainya "Unknown"); Fashionpedia anotasinya CC BY 4.0.
Model hasil latihan Ultralytics = AGPL-3.0 (`docs/adr/0006`). Artefak tugas, bukan untuk diedarkan.

## 1. Runtime

Pelatihan deteksi pada 640 px butuh GPU; sel ini berhenti dengan pesan jelas kalau sesinya CPU.

In [ ]:
# Langkah 1/9 — cek runtime.
!pip install -q ultralytics "datasets>=3" "huggingface_hub>=0.23"

import sys
import torch
import ultralytics

print(f"python        {sys.version.split()[0]}")
print(f"torch         {torch.__version__}")
print(f"ultralytics   {ultralytics.__version__}")
print(f"cuda          {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu           {torch.cuda.get_device_name(0)}")
else:
    print("              panel kanan -> Accelerator -> GPU T4, lalu jalankan ulang dari sel ini")

## 2. Repo

Konverter Fashionpedia dan pemetaan Category-nya ada di skrip repo, jadi notebook ini meng-clone
`feat/ml-pipeline` alih-alih menyalin logikanya.

In [ ]:
# Langkah 2/9 — clone repo.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/kmilaz/gayaku.git"
REPO_REF = "feat/ml-pipeline"
REPO_DIR = Path("/kaggle/temp/gayaku")
WORK = Path("/kaggle/working")
ARTIFACTS = WORK / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )
print("repo:", REPO_DIR)
print(sorted(p.name for p in (REPO_DIR / "ml_workspace/scripts").glob("*.py")))

## 3. Fashionpedia → YOLO

Parquet aslinya diunduh dari HuggingFace, lalu `fashionpedia_to_yolo.py` memetakan kategori item ke
lima Category aplikasi dan membuang kategori potongan pakaian. Kotak per Category dibatasi 2.000,
seperti run detektor sebelumnya.

In [ ]:
import json
# Langkah 3/9 — unduh dan konversi Fashionpedia.
import sys
from pathlib import Path

from huggingface_hub import snapshot_download

HF_DIR = Path("/kaggle/temp/fashionpedia_hf")
FP_DIR = Path("/kaggle/temp/fashionpedia_yolo")
MAX_BOXES_PER_CATEGORY = 2000

if not (HF_DIR / "dataset_infos.json").is_file():
    snapshot_download(
        repo_id="detection-datasets/fashionpedia",
        repo_type="dataset",
        local_dir=str(HF_DIR),
        allow_patterns=["data/*.parquet", "dataset_infos.json"],
    )
if not (FP_DIR / "manifest.json").is_file():
    subprocess.run(
        [sys.executable, str(REPO_DIR / "ml_workspace/scripts/fashionpedia_to_yolo.py"),
         "--hf-dir", str(HF_DIR), "--out", str(FP_DIR), "--force",
         "--max-boxes-per-category", str(MAX_BOXES_PER_CATEGORY)],
        check=True,
    )
manifest = json.loads((FP_DIR / "manifest.json").read_text(encoding="utf-8"))
print("categories:", manifest["categories"])
for split, stats in manifest["splits"].items():
    print(f"{split:<6}", stats["images_written"], "images", stats["per_category"])

## 4. CFPD → YOLO

Dataset Kaggle itu anotasi PASCAL VOC; sepuluh kelasnya dipetakan ke lima Category, dan split
dibagi **per foto** dengan hash tetap supaya arm A, arm B, dan baseline memakai set uji yang sama.

In [ ]:
# Langkah 4/9 — konversi CFPD (VOC) ke YOLO.
import hashlib
import json
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path

CFPS = list(Path("/kaggle/input").glob("*/colorful_fashion_dataset_for_object_detection"))
if not CFPS:
    raise SystemExit(
        "Dataset CFPD tidak ditemukan. Panel kanan -> Add Input -> "
        "cari 'colorful-fashion-dataset-for-object-detection' lalu tambahkan, kemudian jalankan "
        "ulang sel ini."
    )
CF_SRC = CFPS[0]
CF_DIR = Path("/kaggle/temp/cfpd_yolo")
CLASS_TO_CATEGORY = {
    "bag": "Accessories", "sunglass": "Accessories", "hat": "Accessories",
    "dress": "Tops", "shirt": "Tops", "jacket": "Outerwear",
    "pants": "Bottoms", "shorts": "Bottoms", "skirt": "Bottoms", "shoe": "Shoes",
}
NAMES = ["Tops", "Bottoms", "Outerwear", "Shoes", "Accessories"]


def split_of(name: str) -> str:
    bucket = int(hashlib.sha256(name.encode()).hexdigest()[:8], 16) / 0xFFFFFFFF
    return "train" if bucket < 0.8 else ("val" if bucket < 0.9 else "test")


if CF_DIR.exists():
    shutil.rmtree(CF_DIR)
counts = {}
for xml_path in sorted((CF_SRC / "Annotations").glob("*.xml")):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    size = root.find("size")
    width, height = float(size.find("width").text), float(size.find("height").text)
    rows = []
    for obj in root.findall("object"):
        category = CLASS_TO_CATEGORY.get(obj.find("name").text.strip())
        if category is None:
            continue
        box = obj.find("bndbox")
        xmin, ymin = float(box.find("xmin").text), float(box.find("ymin").text)
        xmax, ymax = float(box.find("xmax").text), float(box.find("ymax").text)
        cx, cy = (xmin + xmax) / 2 / width, (ymin + ymax) / 2 / height
        w, h = (xmax - xmin) / width, (ymax - ymin) / height
        rows.append(f"{NAMES.index(category)} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    if not rows:
        continue
    split = split_of(xml_path.stem)
    counts.setdefault(split, {name: 0 for name in NAMES})
    for row in rows:
        counts[split][NAMES[int(row.split()[0])]] += 1
    for folder in ("images", "labels"):
        (CF_DIR / folder / split).mkdir(parents=True, exist_ok=True)
    shutil.copy2(CF_SRC / "JPEGImages" / f"{xml_path.stem}.jpg", CF_DIR / "images" / split / f"{xml_path.stem}.jpg")
    (CF_DIR / "labels" / split / f"{xml_path.stem}.txt").write_text("\n".join(rows) + "\n", encoding="utf-8")

for split in ("train", "val", "test"):
    print(split, len(list((CF_DIR / "images" / split).glob("*.jpg"))), "images", counts.get(split))

## 5. Dua pohon latih

Arm A dan Arm B dibangun dari Fashionpedia yang sama, 6.000 foto, ditambah hanya pada Arm B.
Set uji keduanya memakai CFPD `test`.

In [ ]:
# Langkah 5/9 — susun arm A dan arm B.
import random
from pathlib import Path

SEED = 20261117
FP_TRAIN = 6000


def link(source: Path, target: Path):
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        return
    try:
        target.hardlink_to(source)
    except OSError:
        shutil.copy2(source, target)


def build(out: Path, with_cfpd: bool):
    if out.exists():
        shutil.rmtree(out)
    fp_images = sorted((FP_DIR / "images/train").glob("*.jpg"))
    picks = random.Random(SEED).sample(fp_images, min(FP_TRAIN, len(fp_images)))
    for image in picks:
        link(image, out / "images/train" / image.name)
        link(FP_DIR / "labels/train" / f"{image.stem}.txt", out / "labels/train" / f"{image.stem}.txt")
    if with_cfpd:
        for image in sorted((CF_DIR / "images/train").glob("*.jpg")):
            link(image, out / "images/train" / image.name)
            link(CF_DIR / "labels/train" / f"{image.stem}.txt", out / "labels/train" / f"{image.stem}.txt")
    for image in sorted((FP_DIR / "images/val").glob("*.jpg")):
        link(image, out / "images/val" / image.name)
        link(FP_DIR / "labels/val" / f"{image.stem}.txt", out / "labels/val" / f"{image.stem}.txt")
    for image in sorted((CF_DIR / "images/test").glob("*.jpg")):
        link(image, out / "images/test" / image.name)
        link(CF_DIR / "labels/test" / f"{image.stem}.txt", out / "labels/test" / f"{image.stem}.txt")
    (out / "data.yaml").write_text(
        f"path: {out}\ntrain: images/train\nval: images/val\ntest: images/test\nnames: {NAMES}\n",
        encoding="utf-8",
    )
    return {
        "train": len(list((out / 'images/train').glob('*.jpg'))),
        "val": len(list((out / 'images/val').glob('*.jpg'))),
        "test": len(list((out / 'images/test').glob('*.jpg'))),
    }


ARM_A = Path("/kaggle/temp/arm_a")
ARM_B = Path("/kaggle/temp/arm_b")
print("arm A (Fashionpedia):", build(ARM_A, with_cfpd=False))
print("arm B (+ CFPD):", build(ARM_B, with_cfpd=True))

## 6. Latih Arm A

`yolov8n`, 640 px, 10 epoch, patience 5. Waktu di T4 sekitar 12–15 menit.

In [ ]:
# Langkah 6/9 — latih arm A.
from pathlib import Path
from ultralytics import YOLO

EPOCHS = 10
IMGSZ = 640
BATCH = 32
PATIENCE = 5
RUNS = Path("/kaggle/temp/runs")


def train(tree: Path, name: str):
    model = YOLO("yolov8n.pt")
    model.train(
        data=str(tree / "data.yaml"), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        patience=PATIENCE, device=0, workers=4, project=str(RUNS), name=name,
        exist_ok=True, plots=True, seed=42,
    )
    best = Path(model.trainer.save_dir) / "weights" / "best.pt"
    print(f"{name} -> {best} ({best.stat().st_size / 1e6:.1f} MB)")
    return best


BEST_A = train(ARM_A, "arm-a-fashionpedia")

## 7. Latih Arm B

Konfigurasi identik; satu-satunya yang berubah adalah 1.880 foto CFPD di dalam data latih.

In [ ]:
# Langkah 7/9 — latih arm B.
BEST_B = train(ARM_B, "arm-b-fashionpedia-cfpd")

## 8. Nilai ketiga model pada set uji CFPD yang sama

`val` ultralytics memberi precision, recall, dan AP50 per Category di IoU 0,5. Arm A dan Arm B
diukur dari bobotnya; baseline diukur dari `.tflite` yang benar-benar dipakai aplikasi.

In [ ]:
# Langkah 8/9 — ukur arm A, arm B, dan baseline.
import json
from pathlib import Path

import numpy as np
from ai_edge_litert.interpreter import Interpreter
from PIL import Image


def val_report(best: Path, tree: Path, tag: str) -> dict:
    model = YOLO(str(best))
    metrics = model.val(
        data=str(tree / "data.yaml"), split="test", imgsz=IMGSZ, batch=BATCH,
        device=0, project=str(RUNS), name=f"test-{tag}", exist_ok=True, plots=True,
    )
    rows = {}
    for index in sorted(metrics.names):
        rows[metrics.names[index]] = {
            "precision": float(metrics.box.p[index]),
            "recall": float(metrics.box.r[index]),
            "ap50": float(metrics.box.ap50[index]),
        }
    rows["all"] = {"ap50": float(metrics.box.map50)}
    for png in Path(metrics.save_dir).glob("confusion_matrix*.png"):
        shutil.copy2(png, ARTIFACTS / f"{tag}_{png.name}")
    return rows


report = {"arm_a": val_report(BEST_A, ARM_A, "arm-a"), "arm_b": val_report(BEST_B, ARM_B, "arm-b")}

# Baseline: the shipped .tflite, matched at IoU 0.5 on the same CFPD test photographs.
DETECTOR = REPO_DIR / "flutter_app/assets/models/garment_detector.tflite"
SIZE, THRESHOLD, DUP = 640, 0.25, 0.70


def iou(a, b):
    x0, y0 = max(a[0], b[0]), max(a[1], b[1])
    x1, y1 = min(a[2], b[2]), min(a[3], b[3])
    if x1 <= x0 or y1 <= y0:
        return 0.0
    inter = (x1 - x0) * (y1 - y0)
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


interpreter = Interpreter(model_path=str(DETECTOR))
interpreter.allocate_tensors()
baseline = {name: {"pred": 0, "matched": 0, "gt": 0} for name in NAMES}
for image_path in sorted((CF_DIR / "images/test").glob("*.jpg")):
    image = Image.open(image_path).convert("RGB")
    width, height = image.size
    scale = SIZE / max(width, height)
    cw, ch = round(width * scale), round(height * scale)
    canvas = Image.new("RGB", (SIZE, SIZE), (114, 114, 114))
    left, top = (SIZE - cw) // 2, (SIZE - ch) // 2
    canvas.paste(image.resize((cw, ch), Image.BILINEAR), (left, top))
    array = (np.asarray(canvas, dtype=np.float32) / 255).transpose(2, 0, 1)[None, ...]
    interpreter.set_tensor(interpreter.get_input_details()[0]["index"], array)
    interpreter.invoke()
    raw = interpreter.get_tensor(interpreter.get_output_details()[0]["index"])[0]

    candidates = []
    for anchor in range(raw.shape[1]):
        scores = raw[4:, anchor]
        cls = int(np.argmax(scores))
        if float(scores[cls]) < THRESHOLD:
            continue
        cx, cy, w, h = (float(v) for v in raw[:4, anchor])
        box = (
            max(0.0, (cx - w / 2) * SIZE - left) / cw,
            max(0.0, (cy - h / 2) * SIZE - top) / ch,
            min(SIZE, (cx + w / 2) * SIZE - left) / cw,
            min(SIZE, (cy + h / 2) * SIZE - top) / ch,
        )
        candidates.append((float(scores[cls]), NAMES[cls], box))
    kept = []
    for score, name, box in sorted(candidates, key=lambda item: -item[0]):
        if any(other == name and iou(other_box, box) > DUP for _, other, other_box in kept):
            continue
        kept.append((score, name, box))

    truth = []
    label_path = CF_DIR / "labels/test" / f"{image_path.stem}.txt"
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        cx, cy, w, h = (float(v) for v in parts[1:])
        truth.append((NAMES[int(parts[0])], (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)))

    for name in NAMES:
        predictions = [(score, box) for score, cat, box in kept if cat == name]
        gt_boxes = [box for cat, box in truth if cat == name]
        baseline[name]["pred"] += len(predictions)
        baseline[name]["gt"] += len(gt_boxes)
        used = set()
        for _, box in predictions:
            best, best_index = 0.0, -1
            for index, gt_box in enumerate(gt_boxes):
                if index in used:
                    continue
                overlap = iou(box, gt_box)
                if overlap > best:
                    best, best_index = overlap, index
            if best >= 0.5:
                used.add(best_index)
                baseline[name]["matched"] += 1

report["baseline"] = {
    name: {
        "recall": row["matched"] / row["gt"] if row["gt"] else float("nan"),
        "precision": row["matched"] / row["pred"] if row["pred"] else float("nan"),
        "gt": row["gt"],
        "pred": row["pred"],
    }
    for name, row in baseline.items()
}

print(f"{'Category':12s}{'baseline R/P':>16}{'arm A R/P':>16}{'arm B R/P':>16}")
for name in NAMES:
    base = report["baseline"][name]
    a = report["arm_a"][name]
    b = report["arm_b"][name]
    print(
        f"{name:12s}{base['recall']:8.3f}/{base['precision']:.3f}"
        f"{a['recall']:8.3f}/{a['precision']:.3f}"
        f"{b['recall']:8.3f}/{b['precision']:.3f}"
    )
(ARTIFACTS / "arms.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

## 9. Laporan dan zip

`report.md` menyimpan ketiga tabel itu plus satu paragraf pembacaan: apakah CFPD menolong sepatu
atau tidak, menurut angkanya sendiri.

In [ ]:
# Langkah 9/9 — laporan dan zip.
import zipfile
from datetime import datetime, timezone

def table(key: str, columns: str) -> str:
    lines = [f"| Category | {columns} |", "| --- | --- |"]
    for name in NAMES:
        row = report[key][name]
        if key == "baseline":
            lines.append(f"| {name} | {row['recall']:.3f} / {row['precision']:.3f} |")
        else:
            lines.append(f"| {name} | {row['recall']:.3f} / {row['precision']:.3f} / {row['ap50']:.3f} |")
    return "\n".join(lines)


shoes_a = report["arm_a"]["Shoes"]["recall"]
shoes_b = report["arm_b"]["Shoes"]["recall"]
verdict = (
    "CFPD menolong sepatu: recall Arm B lebih tinggi."
    if shoes_b > shoes_a
    else "CFPD tidak menolong sepatu pada anggaran ini: recall Arm B tidak lebih tinggi."
)

REPORT = f"""# Detektor v2: Fashionpedia + CFPD

| | |
| --- | --- |
| tanggal (UTC) | {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')} |
| notebook | `ml_workspace/yolo/kaggle_detect_v2.ipynb` |
| repo ref | `{REPO_REF}` |
| latihan | yolov8n detection, {EPOCHS} epoch, {IMGSZ} px, batch {BATCH}, patience {PATIENCE} |
| set uji | CFPD test, {len(list((CF_DIR / 'images/test').glob('*.jpg')))} foto, IoU 0.5 |

## Baseline (`.tflite` yang dipakai aplikasi), recall / precision

{table('baseline', 'recall / precision')}

## Arm A — Fashionpedia saja, recall / precision / AP50

{table('arm_a', 'recall / precision / AP50')}

## Arm B — Fashionpedia + 1.880 foto CFPD, recall / precision / AP50

{table('arm_b', 'recall / precision / AP50')}

## Kesimpulan

Recall sepatu: Arm A {shoes_a:.3f}, Arm B {shoes_b:.3f}. {verdict}

Baseline dilatih 40 epoch pada 43 ribu foto, sedangkan kedua arm hanya {EPOCHS} epoch pada 6 ribu
foto, jadi angka baseline **tidak** sebanding sebagai ukuran mutu — yang sebanding adalah Arm A
melawan Arm B, karena keduanya berbagi anggaran, bobot awal, dan set uji.

## Lisensi

CFPD berasal dari dataset riset (Kaggle: Unknown); Fashionpedia CC BY 4.0 untuk anotasinya. Bobot
Ultralytics = AGPL-3.0 (`docs/adr/0006`). Artefak tugas, bukan untuk diedarkan.
"""
(ARTIFACTS / "report.md").write_text(REPORT, encoding="utf-8")

ZIP = WORK / "gayaku_detect_v2.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(ARTIFACTS.iterdir()):
        if path.is_file():
            archive.write(path, path.name)

print(REPORT)
print(f"zip: {ZIP} ({ZIP.stat().st_size:,} bytes)")